# Model Maxxing — Hito A: APM Qwen2.5-7B → Qwen2.5-1.5B (Kaggle)

**Cómo ejecutarlo:** Settings → Accelerator = **GPU P100** (preferible: ~2× ancho de banda que T4) o **GPU T4 x2**
(las variantes se reparten entre las 2 GPUs), Internet = **On**. Después **Save Version → Save & Run All**
(no hace falta dejar el navegador abierto). Al terminar, descarga `hito_a_apm_results.zip` de **Output**.

- Estimación (no medida): 1,5–3,5 h. Si la sesión se corta, `hito_a_apm_after_dev.zip` conserva el desarrollo.
- Repo y modelos (18,3 GB) en `/tmp`: no cuentan como output. El 7B se borra tras construir los slices.
- Sin Docker: los tests de código se ejecutan con `MM_SANDBOX=subprocess` (límites de recursos, sin red si se puede).
- La configuración pre-registrada es `configs/hito_a_apm.yaml`; el μ se congela en dev (`lock`) **antes** del test.
- Cualquier fallo (incluido `resources`) **detiene** el notebook.

In [ ]:
!nvidia-smi
%cd /tmp
!rm -rf modelmaxxing-prev && git clone -q -b hito-a-apm https://github.com/RubenGGBC/modelmaxxing-prev.git
%cd /tmp/modelmaxxing-prev
!git log --oneline -1
!pip install -q -e . "transformers==5.22.2" "datasets==5.1.0" psutil pyyaml 2>&1 | tail -3

In [ ]:
import os, subprocess, time, torch
os.chdir("/tmp/modelmaxxing-prev")
os.environ.update(PYTHONPATH="src", MM_SANDBOX="subprocess", PYTHONIOENCODING="utf-8", TOKENIZERS_PARALLELISM="false")
CFG = "configs/hito_a_apm.yaml"
NOISE = "Warn|Loading weights|it/s|Fetching|Redirects|cached version|Found the latest|  capa [0-9]+/"
N_GPU = max(torch.cuda.device_count(), 1)
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


def sh(cmd, quiet=True):
    """Ejecuta en bash con pipefail y DETIENE el notebook si el comando falla.
    El filtro de ruido nunca provoca fallo (grep sin coincidencias devuelve 1)."""
    if quiet:
        cmd = f"{cmd} 2>&1 | {{ grep -v -E '{NOISE}' || true; }}"
    t0 = time.time()
    r = subprocess.run(["bash", "-o", "pipefail", "-c", cmd])
    print(f"[{time.time() - t0:.0f} s] rc={r.returncode}")
    if r.returncode != 0:
        raise RuntimeError(f"falló (rc={r.returncode}): {cmd}")


def phase(name, extra=""):
    sh(f"python scripts/hito_a_apm.py {CFG} {name} {extra}")


def variants(stage):
    out = subprocess.run(["python", "scripts/hito_a_apm.py", CFG, stage, "--list-variants"],
                         capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
    return out.split()


def parallel(stage):
    """Reparte las variantes entre las GPUs: un proceso por GPU (CUDA_VISIBLE_DEVICES)."""
    vs = variants(stage)
    groups = [g for g in (vs[i::N_GPU] for i in range(N_GPU)) if g]
    procs, t0 = [], time.time()
    for g, group in enumerate(groups):
        print(f"GPU {g}: {group}")
        log = open(f"/tmp/{stage}_gpu{g}.log", "w")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        procs.append((subprocess.Popen(["python", "scripts/hito_a_apm.py", CFG, stage, "--only", *group],
                                       stdout=log, stderr=subprocess.STDOUT, env=env), log, g))
    for p, log, g in procs:
        p.wait()
        log.close()
        print(f"--- GPU {g} (rc={p.returncode}), últimas líneas:")
        print("\n".join(l for l in open(f"/tmp/{stage}_gpu{g}.log").read().splitlines()
                        if "Loading weights" not in l)[-3000:])
    print(f"{stage}: {time.time() - t0:.0f} s")
    bad = [g for p, _, g in procs if p.returncode != 0]
    if bad:
        raise RuntimeError(f"{stage} falló en GPU {bad}: ver /tmp/{stage}_gpu*.log")


def save_zip(name):
    sh(f"zip -qr /kaggle/working/{name} results/experiments/hito_a_apm -x '*/model.safetensors' "
       f"&& ls -la /kaggle/working", quiet=False)


## 1. Tests y recursos (antes de descargar nada)

In [ ]:
sh("python -m pytest -q 2>&1 | tail -3", quiet=False)
phase("resources")   # falla aquí si no hay disco/RAM o el sandbox no funciona

## 2. Descarga con revisiones fijadas (18,3 GB) y nueva comprobación

In [ ]:
sh("python scripts/download_models.py hito_a")
phase("resources")

## 3. Perfilado del donante 7B, slices IM/APM y NLL de los slices

In [ ]:
phase("profile")
phase("slice")
phase("slice_nll")
sh("rm -rf models_cache/Qwen2.5-7B && df -h /tmp", quiet=False)   # copia descargada; ya no hace falta

## 4. Desarrollo: R + IM×μ + APM×μ (elección de μ)

In [ ]:
parallel("dev")
save_zip("hito_a_apm_after_dev.zip")

## 5. Congelar μ (ANTES del test), test sellado e informe

In [ ]:
phase("lock")
sh("cat results/experiments/hito_a_apm/locked_choice.json | head -20", quiet=False)
parallel("test")
phase("report")
save_zip("hito_a_apm_results.zip")